# edge-tts 테스트 — Microsoft 고품질 힌디어 음성 (무료, 인증 불필요)

`aṅguli` 등 5개 단어로 빠알리어 발음 품질 확인

In [ ]:
!pip install -q edge-tts
print('설치 완료!')

In [ ]:
# 데바나가리 변환 함수
CONSONANTS = {
    'kh': '\u0916', 'k': '\u0915', 'gh': '\u0918', 'g': '\u0917', '\u1e45': '\u0919',
    'ch': '\u091b', 'c': '\u091a', 'jh': '\u091d', 'j': '\u091c', '\u00f1': '\u091e',
    '\u1e6dh': '\u0920', '\u1e6d': '\u091f', '\u1e0dh': '\u0922', '\u1e0d': '\u0921', '\u1e47': '\u0923',
    'th': '\u0925', 't': '\u0924', 'dh': '\u0927', 'd': '\u0926', 'n': '\u0928',
    'ph': '\u092b', 'p': '\u092a', 'bh': '\u092d', 'b': '\u092c', 'm': '\u092e',
    'y': '\u092f', 'r': '\u0930', 'l': '\u0932', '\u1e37': '\u0933',
    'v': '\u0935', 's': '\u0938', 'h': '\u0939',
}
VOWELS_IND = { '\u0101': '\u0906', 'a': '\u0905', '\u012b': '\u0908', 'i': '\u0907', '\u016b': '\u090a', 'u': '\u0909', 'e': '\u090f', 'o': '\u0913' }
VOWELS_DEP = { '\u0101': '\u093e', 'a': '', '\u012b': '\u0940', 'i': '\u093f', '\u016b': '\u0942', 'u': '\u0941', 'e': '\u0947', 'o': '\u094b' }
VIRAMA = '\u094d'

def pali_to_devanagari(roman):
    result = ''; i = 0; s = roman.lower()
    while i < len(s):
        ch = s[i]
        if ch in ' ,.;:!?-\n\r\t': result += ch; i += 1; continue
        if ch == '\u1e43': result += '\u0902'; i += 1; continue
        three = s[i:i+3]; two = s[i:i+2]
        consonant = None; consumed = 0
        if three in CONSONANTS: consonant = CONSONANTS[three]; consumed = 3
        elif two in CONSONANTS: consonant = CONSONANTS[two]; consumed = 2
        elif ch in CONSONANTS: consonant = CONSONANTS[ch]; consumed = 1
        if consonant:
            i += consumed
            if i < len(s) and s[i] in VOWELS_IND: result += consonant + VOWELS_DEP[s[i]]; i += 1
            else: result += consonant + VIRAMA
            continue
        if ch in VOWELS_IND: result += VOWELS_IND[ch]; i += 1; continue
        result += ch; i += 1
    return result

print('변환 함수 준비')

In [ ]:
# edge-tts: 힌디어 남성 음성 (느린 속도)
import edge_tts, asyncio, IPython.display as ipd

VOICE = 'hi-IN-MadhurNeural'  # 힌디어 남성
RATE = '-30%'  # 30% 감속

async def speak(text, filename):
    comm = edge_tts.Communicate(text, VOICE, rate=RATE)
    await comm.save(filename)

test_words = ['aṅguli', 'buddha', 'dhammaṃ', 'bhikkhu', 'nibbāna',
              'Buddhaṃ saraṇaṃ gacchāmi', 'sabbe sattā sukhitā hontu']

for word in test_words:
    deva = pali_to_devanagari(word)
    fname = f'{word.replace(" ", "_")}.mp3'
    print(f'\n{word} → {deva}')
    await speak(deva, fname)
    ipd.display(ipd.Audio(fname))